# VGGT view-count reconstruction ablation

SPAR manifests (`dataset/convert_spar_sftqa.py:sample_frames`) cycle-pad scenes with fewer than `num_frames` images, and uniform-sample long scenes without regard to real pose diversity. This notebook tests the direct effect on VGGT geometry: for one or more scenes, sample 1..8 frames (same uniform rule as `sample_frames`) and run frozen VGGT, then compare depth confidence + camera-pose diversity as a function of view count.

Edit the `SCENE_DIRS` / `FRAME_COUNTS` cell below, then run top to bottom.

In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
VGGT_ROOT = ROOT / "vggt"
for p in (ROOT, VGGT_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

print("root:", ROOT)

root: /glob/g01-cache/pf/Yushuo/vjepa201


In [6]:
# ── config — edit this ──────────────────────────────────────────────────────
# spar/{source}/images/{scene_id}/image_color/  <- RGB frames
# siblings: image_depth/ (GT depth, 16-bit mm), pose/ (GT camera-to-world, 4x4 txt) —
# raw ScanNet SensReader layout.
SPAR_ROOT = ROOT / "source_data/spar"
SCENE_DIRS = [
    SPAR_ROOT / "scannet/images/scene0002_00/image_color",
    # add more, e.g. SPAR_ROOT / "scannet/images/scene0002_01/image_color"
]
FRAME_COUNTS = [1, 2, 3, 4, 5, 6, 7, 8]   # never test N > 8; scenes with fewer
                                          # available frames are cycle-padded, not skipped
VGGT_CKPT = ROOT / "ckpts/vggt.pt"
CONF_THRES = 5.0                          # depth_conf threshold for a "valid" pixel
OUTPUT_DIR = ROOT / "outputs/vggt_view_ablation"
SAVE_POINTCLOUD = False
IMG_EXTS = (".jpg", ".jpeg", ".png")
MIN_VALID_DEPTH_PIXELS = 100               # below this, skip depth-error for that frame

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.bfloat16 if device.type == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
print("device:", device, "dtype:", dtype)

device: cpu dtype: torch.float16


## Helpers

- `sample_frames` mirrors `dataset/convert_spar_sftqa.py:sample_frames` **exactly**, including the cycle-pad branch: scenes with fewer than N frames get existing frames duplicated to fill N, same as training data generation.
- Depth error uses the standard monocular-depth protocol (VGGT is up-to-scale): per-frame median-scale alignment against GT, then AbsRel / RMSE / δ&lt;1.25. GT depth is resized+cropped to match VGGT's own `load_and_preprocess_images(mode="crop")` grid — no intrinsics needed for this, it's pure pixel-grid alignment.
- Pose error needs an extra step: VGGT's world frame is arbitrary (its own origin/scale), not ScanNet's metric world. `umeyama_alignment` (closed-form SVD, Umeyama 1991) finds the best similarity transform (rotation + scale + translation) mapping VGGT camera centers onto GT camera centers, then ATE (translation, meters, real scale after alignment) and rotation error (degrees, scale-invariant) are computed in the aligned frame. Rotation error needs ≥3 non-collinear points to be well-conditioned — skipped below that.

In [4]:
# ── frame sampling (mirrors dataset/convert_spar_sftqa.py:sample_frames exactly,
#    cycle-pad branch included — scenes short on frames get duplicates, on purpose) ──
def sample_frames(items: list, num_frames: int) -> list:
    if not items:
        return []
    if len(items) >= num_frames:
        if num_frames == 1:
            return [items[len(items) // 2]]
        idxs = [round(i * (len(items) - 1) / (num_frames - 1)) for i in range(num_frames)]
        return [items[i] for i in idxs]
    return [items[i % len(items)] for i in range(num_frames)]  # cycle-pad


def list_scene_frames(color_dir: Path) -> list[tuple[str, str]]:
    """[(frame_id, path), ...] sorted numerically — filenames are bare ints (0, 1020, 1064, ...),
    not zero-padded, so a plain string sort would put "200" before "30"."""
    paths = [p for p in color_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    paths.sort(key=lambda p: int(p.stem))
    return [(p.stem, str(p)) for p in paths]


def camera_centers(extrinsic: np.ndarray) -> np.ndarray:
    """extrinsic: (S, 3, 4) world-to-cam [R|t] -> (S, 3) camera centers in world coords."""
    R = extrinsic[:, :3, :3]
    t = extrinsic[:, :3, 3]
    return -np.einsum("sij,si->sj", R, t)  # C = -R^T t, per-frame


# ── GT loaders (raw ScanNet SensReader layout) ────────────────────────────────
def load_gt_pose(scene_root: Path, frame_id: str) -> np.ndarray | None:
    """pose/{id}.txt: 4x4 camera-to-world, meters. None if the frame's pose is invalid
    (ScanNet marks some frames with inf/nan — a known artifact, not a bug here)."""
    mat = np.loadtxt(scene_root / "pose" / f"{frame_id}.txt")
    if not np.all(np.isfinite(mat)):
        return None
    return mat


def load_gt_depth_meters(scene_root: Path, frame_id: str) -> np.ndarray:
    """image_depth/{id}.png: 16-bit PNG, millimeters. 0 = invalid/no return."""
    depth_mm = np.array(Image.open(scene_root / "image_depth" / f"{frame_id}.png"))
    return depth_mm.astype(np.float32) / 1000.0


def vggt_crop_geometry(orig_w: int, orig_h: int, target: int = 518) -> tuple[int, int, int, int]:
    """Replicates load_and_preprocess_images(mode="crop")'s resize+crop geometry exactly,
    so GT depth can be resampled onto the same pixel grid as VGGT's predicted depth."""
    new_w = target
    new_h = round(orig_h * (target / orig_w) / 14) * 14
    crop_y0, out_h = 0, new_h
    if new_h > target:
        crop_y0 = (new_h - target) // 2
        out_h = target
    return new_w, new_h, crop_y0, out_h


def resize_gt_depth_to_vggt(gt_depth_m: np.ndarray, orig_w: int, orig_h: int) -> np.ndarray:
    new_w, new_h, crop_y0, out_h = vggt_crop_geometry(orig_w, orig_h)
    t = torch.from_numpy(gt_depth_m)[None, None]  # (1, 1, H, W)
    resized = F.interpolate(t, size=(new_h, new_w), mode="nearest")[0, 0].numpy()
    return resized[crop_y0:crop_y0 + out_h, :]


def umeyama_alignment(src: np.ndarray, dst: np.ndarray, with_scale: bool = True):
    """Closed-form similarity transform (R, t, s) minimizing ||s*R@src + t - dst||^2.
    src, dst: (N, 3). Returns rotation (3,3), translation (3,), scale (float)."""
    n = src.shape[0]
    mu_src, mu_dst = src.mean(axis=0), dst.mean(axis=0)
    src_c, dst_c = src - mu_src, dst - mu_dst
    cov = (dst_c.T @ src_c) / n
    U, D, Vt = np.linalg.svd(cov)
    S = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        S[2, 2] = -1
    R = U @ S @ Vt
    if with_scale:
        var_src = (src_c ** 2).sum() / n
        s = np.trace(np.diag(D) @ S) / var_src
    else:
        s = 1.0
    t = mu_dst - s * R @ mu_src
    return R, t, s


@torch.no_grad()
def run_vggt(model: VGGT, image_paths: list[str]) -> dict:
    images = load_and_preprocess_images(image_paths).to(device)  # (S, 3, H, W)
    with torch.autocast(device_type=device.type, dtype=dtype, enabled=device.type == "cuda"):
        predictions = model(images)
    extrinsic, intrinsic = pose_encoding_to_extri_intri(predictions["pose_enc"], images.shape[-2:])
    predictions["extrinsic"] = extrinsic
    predictions["intrinsic"] = intrinsic
    # do not overwrite predictions["images"] — model.forward() already stores the
    # correctly-batched (1,S,3,H,W) version internally; squeeze(0) below depends on that
    return {k: (v.squeeze(0).float().cpu().numpy() if isinstance(v, torch.Tensor) else v)
            for k, v in predictions.items()}


def save_pointcloud(preds: dict, conf_thres: float, out_path: Path) -> int:
    import trimesh

    pts = preds["world_points"]          # (S, H, W, 3)
    conf = preds["world_points_conf"]    # (S, H, W)
    imgs = preds["images"]               # (S, 3, H, W) in [0, 1]

    mask = conf > conf_thres
    verts = pts[mask]
    if verts.shape[0] == 0:
        return 0
    colors = (imgs.transpose(0, 2, 3, 1)[mask] * 255).astype(np.uint8)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    trimesh.PointCloud(verts, colors=colors).export(str(out_path))
    return int(verts.shape[0])

## Load VGGT (once)

In [5]:
print(f"loading VGGT from {VGGT_CKPT} ...")
model = VGGT()
state = torch.load(VGGT_CKPT, map_location="cpu", weights_only=True)
model.load_state_dict(state)
model.eval().to(device)
print("loaded.")

loading VGGT from /glob/g01-cache/pf/Yushuo/vjepa201/ckpts/vggt.pt ...
loaded.


## Run ablation: 1..8 views per scene, against GT depth + pose

For each scene dir, for each N in `FRAME_COUNTS` (always 1..8 — scenes short on frames get cycle-padded, never skipped), sample N frames and run VGGT, then score against ScanNet GT: depth error (AbsRel/RMSE/δ&lt;1.25, median-scale aligned) and pose error (ATE + rotation, Umeyama-aligned). `world_conf_mean` is kept alongside as a sanity check — it should track actual error, not replace it.

In [7]:
rows = []

for scene_dir in SCENE_DIRS:
    scene_path = Path(scene_dir)
    scene_root = scene_path.parent   # .../scene0002_00  (parent of image_color/)
    frames = list_scene_frames(scene_path)
    n_available = len(frames)
    print(f"\n=== {scene_root.name}: {n_available} frames available ===")

    for n in FRAME_COUNTS:
        sampled = sample_frames(frames, n)          # [(frame_id, path), ...], cycle-padded if n > n_available
        sampled_ids = [f[0] for f in sampled]
        sampled_paths = [f[1] for f in sampled]
        is_padded = n > n_available

        preds = run_vggt(model, sampled_paths)

        # ── depth error vs GT (per-frame median-scale align, then AbsRel/RMSE/delta1) ──
        abs_rels, rmses, delta1s, conf_used = [], [], [], []
        for i, fid in enumerate(sampled_ids):
            orig_w, orig_h = Image.open(sampled_paths[i]).size
            gt_depth = resize_gt_depth_to_vggt(load_gt_depth_meters(scene_root, fid), orig_w, orig_h)
            pred_depth = preds["depth"][i, ..., 0]
            conf = preds["depth_conf"][i]
            mask = (gt_depth > 0) & (conf > CONF_THRES)
            if mask.sum() < MIN_VALID_DEPTH_PIXELS:
                continue
            scale = np.median(gt_depth[mask]) / np.median(pred_depth[mask])
            pred_scaled = pred_depth * scale
            abs_rels.append(float(np.mean(np.abs(pred_scaled[mask] - gt_depth[mask]) / gt_depth[mask])))
            rmses.append(float(np.sqrt(np.mean((pred_scaled[mask] - gt_depth[mask]) ** 2))))
            thresh = np.maximum(pred_scaled[mask] / gt_depth[mask], gt_depth[mask] / pred_scaled[mask])
            delta1s.append(float(np.mean(thresh < 1.25)))
            conf_used.append(float(conf[mask].mean()))

        # ── pose error vs GT (Umeyama-align VGGT camera centers onto GT, then ATE + rotation) ──
        gt_poses = [load_gt_pose(scene_root, fid) for fid in sampled_ids]
        valid_idx = [i for i, p in enumerate(gt_poses) if p is not None]
        ate_rmse_m, mean_rot_err_deg = float("nan"), float("nan")
        if len(valid_idx) >= 2:
            gt_centers = np.stack([gt_poses[i][:3, 3] for i in valid_idx])
            vggt_centers = camera_centers(preds["extrinsic"])[valid_idx]
            R_align, t_align, s_align = umeyama_alignment(vggt_centers, gt_centers, with_scale=True)
            aligned_centers = (s_align * (R_align @ vggt_centers.T).T) + t_align
            ate_rmse_m = float(np.sqrt(np.mean(np.sum((aligned_centers - gt_centers) ** 2, axis=1))))

            if len(valid_idx) >= 3:
                gt_rots = np.stack([gt_poses[i][:3, :3] for i in valid_idx])
                vggt_R_c2w = np.transpose(preds["extrinsic"][:, :3, :3], (0, 2, 1))[valid_idx]
                rot_errs = []
                for i in range(len(valid_idx)):
                    R_rel = gt_rots[i].T @ (R_align @ vggt_R_c2w[i])
                    cos_a = np.clip((np.trace(R_rel) - 1.0) / 2.0, -1.0, 1.0)
                    rot_errs.append(np.degrees(np.arccos(cos_a)))
                mean_rot_err_deg = float(np.mean(rot_errs))

        row = {
            "scene": scene_root.name,
            "n_views": n,
            "n_available_frames": n_available,
            "is_padded": is_padded,
            "depth_abs_rel": float(np.mean(abs_rels)) if abs_rels else float("nan"),
            "depth_rmse_m": float(np.mean(rmses)) if rmses else float("nan"),
            "depth_delta1": float(np.mean(delta1s)) if delta1s else float("nan"),
            "n_frames_depth_scored": len(abs_rels),
            "pose_ate_rmse_m": ate_rmse_m,
            "pose_mean_rot_err_deg": mean_rot_err_deg,
            "n_frames_pose_scored": len(valid_idx),
            "world_conf_mean": float(np.mean(conf_used)) if conf_used else float("nan"),
            "frames": sampled_ids,
        }

        if SAVE_POINTCLOUD:
            ply_path = OUTPUT_DIR / scene_root.name / f"n{n:02d}{'_pad' if is_padded else ''}.ply"
            save_pointcloud(preds, CONF_THRES, ply_path)
            row["pointcloud"] = str(ply_path)

        rows.append(row)
        print(f"n={n}{'(pad)' if is_padded else '':4}: AbsRel={row['depth_abs_rel']:.3f} "
              f"RMSE={row['depth_rmse_m']:.3f}m d1={row['depth_delta1']:.3f}  "
              f"ATE={ate_rmse_m:.3f}m rot_err={mean_rot_err_deg:.2f}deg")

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
df


=== scene0002_00: 226 frames available ===
n=1    : AbsRel=0.029 RMSE=0.046m d1=1.000  ATE=nanm rot_err=nandeg
n=2    : AbsRel=0.020 RMSE=0.048m d1=0.999  ATE=0.000m rot_err=nandeg
n=3    : AbsRel=0.014 RMSE=0.030m d1=1.000  ATE=0.007m rot_err=2.49deg
n=4    : AbsRel=0.018 RMSE=0.042m d1=0.999  ATE=0.044m rot_err=3.36deg
n=5    : AbsRel=0.014 RMSE=0.030m d1=0.999  ATE=0.214m rot_err=4.94deg
n=6    : AbsRel=0.018 RMSE=0.034m d1=1.000  ATE=0.200m rot_err=6.02deg


KeyboardInterrupt: 

## Plot: metric vs. view count (per scene)

In [ ]:
import matplotlib.pyplot as plt

metrics_to_plot = ["depth_abs_rel", "depth_rmse_m", "depth_delta1",
                    "pose_ate_rmse_m", "pose_mean_rot_err_deg", "world_conf_mean"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))

for ax, metric in zip(axes.flat, metrics_to_plot):
    for scene, g in df.groupby("scene"):
        real = g[~g["is_padded"]]
        pad = g[g["is_padded"]]
        ax.plot(real["n_views"], real[metric], marker="o", label=f"{scene} (real)")
        if len(pad):
            ax.plot(pad["n_views"], pad[metric], marker="x", linestyle="--", label=f"{scene} (padded)")
    ax.set_xlabel("n_views")
    ax.set_ylabel(metric)
    ax.set_title(metric)
    ax.grid(alpha=0.3)

axes.flat[0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "metrics_vs_n_views.png", dpi=150)
plt.show()

## Visualize: best vs worst vs GT (interactive, plotly)

Picks the best/worst row (by `depth_abs_rel`) for one scene, loads their saved `.ply`, and builds an independent GT point cloud by unprojecting GT depth through GT pose + `intrinsic/intrinsic_color.txt` (raw ScanNet layout, top-left 3x3 of the 4x4 = K).

Caveat: the three panels are **not** in the same coordinate frame — VGGT's world is up-to-scale/arbitrary-origin (same issue as the pose-error alignment above), the GT cloud is in ScanNet's real metric frame. Each subplot auto-scales independently, so this is for eyeballing structure/density/noise, not overlay/registration. If you want a true overlay, reuse `umeyama_alignment` on that row's camera centers first.

In [12]:
!pip install plotly

Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 25.4 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [plotly]  WARNING: The script plotly_get_chrome is installed in '/home/go98qit/.local/bin' which is not on PATH.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [plotly]2m1/2 [plotly]

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [1]:
import trimesh
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def load_ply_points(path, max_points: int = 50000):
    pc = trimesh.load(str(path))
    pts, colors = pc.vertices, pc.colors[:, :3]
    if len(pts) > max_points:
        idx = np.random.choice(len(pts), max_points, replace=False)
        pts, colors = pts[idx], colors[idx]
    return pts, colors


def load_intrinsic_color(scene_root: Path) -> np.ndarray:
    """intrinsic/intrinsic_color.txt: 4x4, raw ScanNet layout — top-left 3x3 is K."""
    mat = np.loadtxt(scene_root / "intrinsic" / "intrinsic_color.txt")
    return mat[:3, :3]


def adjust_intrinsic_for_crop(K: np.ndarray, orig_w: int, orig_h: int) -> np.ndarray:
    new_w, new_h, crop_y0, out_h = vggt_crop_geometry(orig_w, orig_h)
    sx, sy = new_w / orig_w, new_h / orig_h
    K2 = K.astype(np.float64).copy()
    K2[0, 0] *= sx; K2[0, 2] *= sx
    K2[1, 1] *= sy; K2[1, 2] = K2[1, 2] * sy - crop_y0
    return K2


def resize_color_like_vggt(img: Image.Image, orig_w: int, orig_h: int) -> np.ndarray:
    new_w, new_h, crop_y0, out_h = vggt_crop_geometry(orig_w, orig_h)
    arr = np.array(img.resize((new_w, new_h), Image.Resampling.BICUBIC))
    return arr[crop_y0:crop_y0 + out_h, :, :]


def gt_pointcloud_for_row(row, max_points: int = 50000):
    """Unprojects GT depth (via GT pose + intrinsic_color) into ScanNet's metric world
    frame — independent of VGGT's own frame, see markdown caveat above."""
    scene_root = SPAR_ROOT / "scannet/images" / row["scene"]
    K = load_intrinsic_color(scene_root)
    all_pts, all_colors = [], []
    for fid in row["frames"]:
        img = Image.open(scene_root / "image_color" / f"{fid}.jpg")
        orig_w, orig_h = img.size
        depth = resize_gt_depth_to_vggt(load_gt_depth_meters(scene_root, fid), orig_w, orig_h)
        pose = load_gt_pose(scene_root, fid)
        if pose is None:
            continue
        K_adj = adjust_intrinsic_for_crop(K, orig_w, orig_h)
        H, W = depth.shape
        ys, xs = np.mgrid[0:H, 0:W]
        x = (xs - K_adj[0, 2]) * depth / K_adj[0, 0]
        y = (ys - K_adj[1, 2]) * depth / K_adj[1, 1]
        pts_cam = np.stack([x, y, depth], axis=-1)
        R, t = pose[:3, :3], pose[:3, 3]
        pts_world = pts_cam @ R.T + t
        color = resize_color_like_vggt(img, orig_w, orig_h)
        mask = depth > 0
        all_pts.append(pts_world[mask])
        all_colors.append(color[mask])
    pts, colors = np.concatenate(all_pts), np.concatenate(all_colors)
    if len(pts) > max_points:
        idx = np.random.choice(len(pts), max_points, replace=False)
        pts, colors = pts[idx], colors[idx]
    return pts, colors


def scatter3d(pts, colors, name):
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2], mode="markers", name=name,
        marker=dict(size=1.5, color=[f"rgb({r},{g},{b})" for r, g, b in colors]),
    )


# ── pick best/worst by depth_abs_rel for one scene ──────────────────────────
scene_name = Path(SCENE_DIRS[0]).parent.name
scored = df[(df["scene"] == scene_name) & df["depth_abs_rel"].notna()]
best_row = scored.loc[scored["depth_abs_rel"].idxmin()]
worst_row = scored.loc[scored["depth_abs_rel"].idxmax()]
print(f"best:  n={best_row['n_views']} pad={best_row['is_padded']} AbsRel={best_row['depth_abs_rel']:.3f}")
print(f"worst: n={worst_row['n_views']} pad={worst_row['is_padded']} AbsRel={worst_row['depth_abs_rel']:.3f}")

best_pts, best_colors = load_ply_points(best_row["pointcloud"])
worst_pts, worst_colors = load_ply_points(worst_row["pointcloud"])
gt_pts, gt_colors = gt_pointcloud_for_row(best_row)   # swap to worst_row for the other frame set

fig = make_subplots(rows=1, cols=3, specs=[[{"type": "scene"}] * 3],
                     subplot_titles=(f"best (n={best_row['n_views']})",
                                      f"worst (n={worst_row['n_views']})", "GT"))
fig.add_trace(scatter3d(best_pts, best_colors, "best"), row=1, col=1)
fig.add_trace(scatter3d(worst_pts, worst_colors, "worst"), row=1, col=2)
fig.add_trace(scatter3d(gt_pts, gt_colors, "GT"), row=1, col=3)
fig.update_layout(height=500, width=1400, showlegend=False,
                   scene=dict(aspectmode="data"), scene2=dict(aspectmode="data"), scene3=dict(aspectmode="data"))
fig.show()

NameError: name 'Path' is not defined